In [ ]:
using Distributed, SharedArrays, RCall;R"""library(abc);library(data.table);library(dplyr)""";
addprocs(5)
@everywhere using Analytical, CSV, DataFrames
include("/home/jmurga/mkt/202004/scripts/src/summaryParser.jl")
abcreg = "/home/jmurga/ABCreg/src/reg"

In [ ]:
adap = Analytical.parameters(N=1000,n=661,gamNeg=-457,gL=10,gH=500,bRange=push!(collect(0.1:0.1:0.9),0.999))
Analytical.binomOp!(adap);
dac = [5,10,20,50,100,200,500,1000]

In [ ]:
R"""
performABC = function(df,alpha,tol,method='rejection',transf='none'){
    
    p = df[,1:3]
    s = df[,4:ncol(df)]

    out = abc(param=p, sumstat=s, target=alpha, tol = tol, method=method,transf=transf)
    summary(out)
    return(as.data.table(out[[1]]))
}
"""

# Random DFE

## Whole-genome

In [ ]:
wg = Analytical.parseSfs(sample=661,data="/home/jmurga/mkt/202004/rawData/tgp/wg.tsv",dac=dac)
paramW = wg[1]
amk = Analytical.asympFit(wg[1])[1]

@time wgDf = summaryStats(param = adap,amk = amk, divergence = wg[3], gH=[500,300,200],gL=collect(5:10), sfs = wg[2],dac = dac ,iterations = 10^5, scale=0.000402,shape=0.184,fixed=false);

In [ ]:
CSV.write("/home/jmurga/mkt/202004/rawData/summStat/tgp/wgSummStat.tsv",DataFrame(wgDf),delim='\t',header=false)
CSV.write("/home/jmurga/mkt/202004/rawData/summStat/tgp/wgAlpha.tsv",DataFrame(wg[1]'),delim='\t',header=false)

In [ ]:
reg = `$abcreg -p /home/jmurga/mkt/202004/rawData/summStat/tgp/wgSummStat.tsv -d /home/jmurga/mkt/202004/rawData/summStat/tgp/wgAlpha.tsv -P 3 -S 9  -T -t 0.01 -b  /home/jmurga/mkt/202004/rawData/summStat/tgp/wg`
run(reg)

In [ ]:
R"""
inferWg = fread("/home/jmurga/mkt/202004/rawData/summStat/tgp/wg.0.tangent.post.gz")
summary(df)
"""

## VIPs

In [ ]:
vips = Analytical.parseSfs(sp=661,data="/home/jmurga/mkt/202004/rawData/tgp/vips.tsv",dac=dac)
paramV = vips[1]
amk = Analytical.asympFit(vips[1])[1]

@time vipsDf = Analytical.summaryStats(param = adap, amk = amk, divergence = vips[3],gH=[500,300,200],gL=collect(5:10), sfs = vips[2],dac = dac ,iterations = 10^4,scale=0.000402,shape=0.184,fixed=false);

In [ ]:
CSV.write("/home/jmurga/mkt/202004/rawData/summStat/tgp/vipsSummStat.tsv",DataFrame(vipsDf),delim='\t',header=false)
CSV.write("/home/jmurga/mkt/202004/rawData/summStat/tgp/vipsAlpha.tsv",DataFrame(vips[1]'),delim='\t',header=false)

In [ ]:
reg = `$abcreg -p /home/jmurga/mkt/202004/rawData/summStat/tgp/vipsSummStat.tsv -d /home/jmurga/mkt/202004/rawData/summStat/tgp/vipsAlpha.tsv -P 3 -S 9  -T -t 0.01 -b  /home/jmurga/mkt/202004/rawData/summStat/tgp/vips`
run(reg)

In [ ]:
R"""
inferVips = fread("/home/jmurga/mkt/202004/rawData/summStat/tgp/vips.0.tangent.post.gz")
summary(df)
"""

## Non-VIPs

In [ ]:
nonvips = Analytical.parseSfs(sample=661,data="/home/jmurga/mkt/202004/rawData/tgp/nonvips.tsv",dac=dac)
paramNV = nonvips[1]
amk = Analytical.asympFit(nonvips[1])[1]

@time nonvipsDf = Analytical.summaryStats(param = adap, amk = amk, divergence = vips[3],gH=[500,300,200],gL=collect(5:10), sfs = nonvips[2],dac = dac ,iterations = 10^4,scale=0.000402,shape=0.184,fixed=false);

In [ ]:
CSV.write("/home/jmurga/mkt/202004/rawData/summStat/tgp/nonvipsSummStat.tsv",DataFrame(nonvipsDf),delim='\t',header=false)
CSV.write("/home/jmurga/mkt/202004/rawData/summStat/tgp/nonvipsAlpha.tsv",DataFrame(nonvips[1]'),delim='\t',header=false)

In [ ]:
reg = `$abcreg -p /home/jmurga/mkt/202004/rawData/summStat/tgp/nonvipsSummStat.tsv -d /home/jmurga/mkt/202004/rawData/summStat/tgp/nonvipsAlpha.tsv -P 3 -S 9 -T -t 0.01 -b  /home/jmurga/mkt/202004/rawData/summStat/tgp/nonvips`
run(reg)

In [ ]:
R"""
inferNonvips = fread("/home/jmurga/mkt/202004/rawData/summStat/tgp/nonvips.0.tangent.post.gz")
summary(df)
"""

## Plot

In [ ]:
R"""
inferWg$analysis = "Whole-genome dataset"
inferVips$analysis = "VIPs dataset"
inferNonvips$analysis = "Non-VIPs dataset"
df = rbind(inferWg,inferVips,inferNonvips)
names(df) = c(paste(expression(alpha[w])),paste(expression(alpha[s])),paste(expression(alpha)),'analysis')

dfPlot = melt(df)
    
tgpPlot = ggplot(dfPlot) + geom_density(aes(x=value,fill=variable),alpha=0.75) + 
    facet_wrap(~analysis) + 
    scale_fill_manual("Posterior distribution",values = paletteSanMiguel,labels=c(expression(paste("Posterior ",alpha[w])), expression(paste("Posterior ",alpha[s])),expression(paste("Posterior ",alpha)))) + 
xlab(expression(alpha)) + 
    ylab("") + 
    theme_bw()

fwrite(df,'/home/jmurga/mkt/202004/results/abc/tgp/tgpABC.tsv',sep='\t')
ggsave(tgpPlot,filename='/home/jmurga/mkt/202004/results/abc/tgp/tgpABC.svg',width=10,height=8)


dfQ = dfPlot %>% group_by(analysis,variable) %>% summarize(q=paste0(round(mean(value),3)," [",quantile(round(value,3),c(0.1)),"-",quantile(round(value,3),0.9),"]"))
dfQ = reshape2::dcast(dfQ,analysis~variable)

fwrite(dfQ,'/home/jmurga/mkt/202004/results/abc/tgp/tgpNNABC.tsv',sep='\t')

tgpPlot
"""

## Additional rejection procedure

In [ ]:
R"""
wgRejection = performABC($wgDf,$paramW,tol,'rejection')
vipsRejection = performABC($vipsDf,$paramV,tol,'rejection')
nonvipsRejection = performABC($nonvipsDf,$paramNV,tol,'rejection')

wgRejection[['analysis']] = 'Whole-genome dataset'
vipsRejection[['analysis']] = 'VIPs dataset'
nonvipsRejection[['analysis']] = 'Non-vips dataset'

dfRejection = rbind(wgRejection,vipsRejection,nonvipsRejection)
names(dfRejection) = c(paste(expression(alpha[w])),paste(expression(alpha[s])),paste(expression(alpha)),'analysis')

dfPlotRejection = melt(dfRejection)


tgpPlotRejection = ggplot(dfPlotRejection) + geom_density(aes(x=value,fill=variable),alpha=0.5) + 
    facet_wrap(~analysis) + 
    scale_fill_manual("Posterior distribution",values = paletteSanMiguel,labels=c(expression(paste("Posterior ",alpha[w])), expression(paste("Posterior ",alpha[s])),expression(paste("Posterior ",alpha)))) + 
xlab(expression(alpha)) + 
    ylab("") + 
    theme_bw()

fwrite(df,'/home/jmurga/mkt/202004/results/abc/tgp/tgpRejection.tsv',sep='\t')
ggsave(tgpPlotRejection,filename='/home/jmurga/mkt/202004/results/abc/tgp/tgpRejection.svg')

tgpPlotRejection
"""

# Fixed DFE

## Whole-genome

In [ ]:
wg = Analytical.parseSfs(sample=661,data="/home/jmurga/mkt/202004/rawData/tgp/wg.tsv",dac=dac)
paramW = wg[1]
amk = Analytical.asympFit(wg[1])[1]

@time wgDf = summaryStats(param = adap,amk = amk, divergence = wg[3], sfs = wg[2],dac = dac ,iterations = 10^5, scale=0.000402,shape=0.184,fixed=true);

In [ ]:
R"""
tol = 2000/nrow($wgDf)
wg = performABC($wgDf,$paramW,tol,'rejection')
wg[['analysis']] = 'Whole-genome dataset'
"""

In [ ]:
CSV.write("/home/jmurga/mkt/202004/rawData/summStat/tgp/wgSummStatFixed.tsv",DataFrame(wgDf),delim='\t',header=false)
CSV.write("/home/jmurga/mkt/202004/rawData/summStat/tgp/wgAlpha.tsv",DataFrame(wg[1]'),delim='\t',header=false)

## VIPs

In [ ]:
vips = Analytical.parseSfs(sample=661,data="/home/jmurga/mkt/202004/rawData/tgp/vips.tsv",dac=dac)
paramV = vips[1]
amk = Analytical.asympFit(vips[1])[1]

@time vipsDf = summaryStats(param = adap, amk = amk, divergence = vips[3], sfs = vips[2],dac = dac ,iterations = 10^5,scale=0.000402,shape=0.184,fixed=true);

In [ ]:
R"""
tol = 2000/nrow($vipsDf)
vips = performABC($vipsDf,$paramV,tol,'rejection')
vips[['analysis']] = 'VIPs dataset'
"""

In [ ]:
CSV.write("/home/jmurga/mkt/202004/rawData/summStat/tgp/vipsSummStatFixed.tsv",DataFrame(vipsDf),delim='\t',header=false)
CSV.write("/home/jmurga/mkt/202004/rawData/summStat/tgp/vipsAlphaFixed.tsv",DataFrame(vips[1]),delim='\t',header=false)

## Non-VIPs

In [ ]:
nonvips = Analytical.parseSfs(sample=661,data="/home/jmurga/mkt/202004/rawData/tgp/nonvips.tsv",dac=dac)
paramNV = nonvips[1]
amk = Analytical.asympFit(nonvips[1])[1]

@time nonvipsDf = summaryStats(param = adap, amk = amk, divergence = nonvips[3], sfs = nonvips[2],dac = dac ,iterations = 10^5,scale=0.000402,shape=0.184,fixed=true);

In [ ]:
R"""
tol = 2000/nrow($nonvipsDf)
nonvips = performABC($nonvipsDf,$paramNV,tol,'rejection')
nonvips[['analysis']] = 'Non-VIPs dataset'
"""

In [ ]:
CSV.write("/home/jmurga/mkt/202004/rawData/summStat/tgp/nonvipsSummStatFixed.tsv",DataFrame(nonvipsDf),delim='\t',header=false)
CSV.write("/home/jmurga/mkt/202004/rawData/summStat/tgp/nonvipsAlphaFixed.tsv",DataFrame(nonvips[1]),delim='\t',header=false)

## Plot

In [ ]:
R"""
df = rbind(wg,vips,nonvips)
names(df) = c(paste(expression(alpha[w])),paste(expression(alpha[s])),paste(expression(alpha)),'analysis')

dfPlot = melt(df)
    
tgpPlot = ggplot(dfPlot) + geom_density(aes(x=value,fill=variable),alpha=0.5) + 
    facet_wrap(~analysis) + 
    scale_fill_manual("Posterior distribution",values = paletteSanMiguel,labels=c(expression(paste("Posterior ",alpha[w])), expression(paste("Posterior ",alpha[s])),expression(paste("Posterior ",alpha)))) + 
xlab(expression(alpha)) + 
    ylab("") + 
    theme_bw()

fwrite(df,'/home/jmurga/mkt/202004/results/abc/tgp/tgpABCFixed.tsv',sep='\t')
ggsave(tgpPlot,filename='/home/jmurga/mkt/202004/results/abc/tgp/tgpABCFixed.svg')

tgpPlot
"""

# B analysis

In [ ]:
adap = Analytical.parameters(N=1000,n=661,gamNeg=-457,gL=10,gH=500,bRange=permutedims(collect(0.1:0.1:0.9)))
Analytical.binomOp!(adap);

dac = [2,5,10,20,50,100,200,500,1000]

In [ ]:
rejection = Dict{Float64,Array}()
nn = Dict{Float64,Array}()
asymp = OrderedDict{Float64,Float64}()


output = DataFrame(aw = Float64[],as = Float64[],alpha = Float64[], B = Float64[])
bAnalysis = collect(0.05:0.05:0.9)
bAnalysis = collect(0.1:0.1:0.9)


for b in eachindex(bAnalysis)
    
    bval= bAnalysis[b]
    if  bAnalysis[b] == 0.9
        bval= [0.9 0.999]
    else
        bval=permutedims([bAnalysis[b]])
    end
        
    adap = Analytical.parameters(N=1000,n=661,gamNeg=-457,gL=10,gH=500,bRange=bval,B=bval[1])
    Analytical.binomOp!(adap);
        
    tmp = parseSfs(sp=661,data="/home/jmurga/mkt/202004/rawData/tgp/tgpProteinsB.tsv",dac=collect(1:1000),B=bval[1])
    amk = Analytical.asympFit(tmp[1])[1]
    
    #imk = imputedMK(sfs=tmp[4],divergence=tmp[5])["alpha"]
    
    asymp[bAnalysis[b]] = amk
    
    #@time df = summaryStats(param = adap,amk = amk, divergence = tmp[3], sfs = tmp[2],dac = dac ,iterations = 10^4, scale=0.000402,shape=0.184,fixed=false);
    
    #p = df[:,1:3]
    #s = df[:,4:end]

    #outNN = rcopy(R"""out = suppressWarnings(abc(param=$p,target=$alpha,sumstat=$s,method='neuralnet',tol=0.2))[1:2]""")
    
    #rejection[bAnalysis[b]] = outNN[:unadj_values]
    #nn[bAnalysis[b]] = outNN[:adj_values]
    #push!(output,hcat(mean(nn[bAnalysis[b]],dims=1),bAnalysis[b]))
    
end